In [8]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
from pathlib import Path
import time

# =========================
# 0. LOAD ENV VARIABLE
# =========================
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

MONGO_URL = os.getenv("MONGODB_URL")

if not MONGO_URL:
    raise Exception("MONGODB_URL is not set. Add it to a .env file in the project root.")

print("Mongo URL loaded")

# =========================
# 1. LOAD DATA
# =========================
csv_path = PROJECT_ROOT / "notebook" / "LOAN.csv"
if not csv_path.exists():
    csv_path = PROJECT_ROOT / "LOAN.csv"

if not csv_path.exists():
    raise FileNotFoundError("LOAN.csv not found in project root or notebook folder")

df = pd.read_csv(csv_path)
print("Original shape:", df.shape)

# =========================
# 2. CLEAN DATA
# =========================
if "ID" in df.columns:
    df = df.drop_duplicates(subset=["ID"])

# Convert object columns to string
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str)

df = df.convert_dtypes()
print("After cleaning:", df.shape)

data = df.to_dict(orient="records")

# =========================
# 3. CONNECT MONGODB
# =========================
client = MongoClient(MONGO_URL, serverSelectionTimeoutMS=5000)

# Test connection
client.server_info()
print("MongoDB Connected")

print("Databases BEFORE insert:", client.list_database_names())

# =========================
# 4. DB + COLLECTION
# =========================
db = client["Proj"]
collection = db["loan_data"]

# =========================
# 5. CLEAR OLD DATA
# =========================
collection.delete_many({})
print("Old data cleared")

# =========================
# 6. INSERT DATA (BATCH)
# =========================
batch_size = 1000

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]

    for attempt in range(3):
        try:
            collection.insert_many(batch, ordered=False)
            print(f"Inserted {i} to {i + len(batch)}")
            break
        except Exception as e:
            print(f"Retry {attempt + 1}: {e}")
            time.sleep(2)

# =========================
# 7. VERIFY INSERTION
# =========================
count = collection.count_documents({})
print("Total documents inserted:", count)

print("Databases AFTER insert:", client.list_database_names())

collection.insert_one({"status": "verified"})

print("Data inserted successfully!")

✅ Mongo URL loaded
Original shape: (148670, 34)
After cleaning: (148670, 34)
✅ MongoDB Connected
Databases BEFORE insert: ['notes', 'sample_mflix', 'admin', 'local']
🧹 Old data cleared
Inserted 0 → 1000
Inserted 1000 → 2000
Inserted 2000 → 3000
Inserted 3000 → 4000
Inserted 4000 → 5000
Inserted 5000 → 6000
Inserted 6000 → 7000
Inserted 7000 → 8000
Inserted 8000 → 9000
Inserted 9000 → 10000
Inserted 10000 → 11000
Inserted 11000 → 12000
Inserted 12000 → 13000
Inserted 13000 → 14000
Inserted 14000 → 15000
Inserted 15000 → 16000
Inserted 16000 → 17000
Inserted 17000 → 18000
Inserted 18000 → 19000
Inserted 19000 → 20000
Inserted 20000 → 21000
Inserted 21000 → 22000
Inserted 22000 → 23000
Inserted 23000 → 24000
Inserted 24000 → 25000
Inserted 25000 → 26000
Inserted 26000 → 27000
Inserted 27000 → 28000
Inserted 28000 → 29000
Inserted 29000 → 30000
Inserted 30000 → 31000
Inserted 31000 → 32000
Inserted 32000 → 33000
Inserted 33000 → 34000
Inserted 34000 → 35000
Inserted 35000 → 36000
Inserted 